# 基本編：データの読み込みとGPS可視化

このノートブックでは、libxrkを使用してAIM™テレメトリーデータを読み込み、可視化する基本を説明します。

## このノートブックの内容

- **データの読み込み**: AIMソフトウェアなしで`.xrk`または`.xrz`ファイルを読み込み
- **ラップタイム一覧**: 記録されたすべてのラップとラップタイムを表示
- **GPS速度マップ**: インタラクティブな地図上にコース全体の速度を可視化
- **ブレーキ＆スロットルオーバーレイ**: ドライバー入力をコースマップ上に重ねて表示

## 自分のデータを使用する場合

自分のデータを分析するには：

1. 下の**最初のセルを実行**してパッケージをインストールし、アップロードウィジェットを表示
2. **「Choose File」をクリック**して`.xrk`または`.xrz`ファイルを選択
3. **残りのセルをすべて実行**してデータを分析

ステータスインジケーターに使用中のファイルが表示されます。ファイルをアップロードしない場合は、サンプルデータが使用されます。

## 必要なチャンネル

- GPSデータチャンネル（`GPS Latitude`、`GPS Longitude`、`GPS Speed`）
- 入力オーバーレイ用のブレーキ圧（`BrakePress`）とスロットル（`PPS`）

**注意:** このノートブックはJupyterLite（ブラウザ）と通常のJupyterLab環境の両方で動作します。

In [ ]:
# 必要なパッケージをインストール（JupyterLiteで必要、通常のJupyterLabでは既にインストール済みならスキップ）
%pip install -q pandas plotly libxrk motorsports-data-notebook jinja2 ipywidgets

# ヘルパー関数をインポート
from motorsports_data_notebook.visualization import (
    format_lap_time,
    plot_gps_channels,
    show_fig,
)
from motorsports_data_notebook.widgets import SessionPicker

# セッションピッカーとチャンネル設定
# 自分のファイルをアップロードして分析するラップを選択
session = SessionPicker(
    default_file="../data/CMD_Inferno 86_Fuji GP Sh_Generic testing_a_2248.xrz",
    channel_mapping={
        "gps_latitude": "GPS Latitude",
        "gps_longitude": "GPS Longitude",
        "throttle": "PPS",
        "brake": "BrakePress",
    },
)
session.display()

In [ ]:
# ラップ情報をpandas DataFrameとして取得
laps = session.get_laps()

In [ ]:
# ラップタイム一覧を表示
laps.style.format({"lap_time": format_lap_time})  # type: ignore[dict-item]

In [ ]:
# libxrk 0.5.0のメソッドを使用して選択したラップのチャンネルデータを抽出
selected_lap = session.get_selected_lap()
log = session.get_log()
CHANNEL_NAMES = session.get_channel_names()
lap_num = int(selected_lap["num"])

# 設定からチャンネル名を取得
gps_lat_ch = CHANNEL_NAMES["gps_latitude"]
gps_lon_ch = CHANNEL_NAMES["gps_longitude"]
throttle_ch = CHANNEL_NAMES["throttle"]
brake_ch = CHANNEL_NAMES["brake"]

# ラップでフィルタし、チャンネルを選択し、GPS時間軸にリサンプル
channels = (
    log.filter_by_lap(lap_num)
    .select_channels([gps_lat_ch, gps_lon_ch, "speed_kmh", brake_ch, throttle_ch])
    .resample_to_channel(gps_lat_ch)
    .channels
)

In [ ]:
# チャンネルテーブルを直接使用してGPSマップ上に速度をプロット
fig = plot_gps_channels(
    channels,
    lat_channel=gps_lat_ch,
    lon_channel=gps_lon_ch,
    color_channels=[("speed_kmh", "速度 (km/h)", "Viridis")],
    title=f"速度 - ラップ {int(selected_lap['num'])}",
)
show_fig(fig)

In [ ]:
# 複数のカラーチャンネルでプロット（GPS時間軸に自動補間）
fig = plot_gps_channels(
    channels,
    lat_channel=gps_lat_ch,
    lon_channel=gps_lon_ch,
    color_channels=[
        (brake_ch, "ブレーキ圧", "Reds"),
        (throttle_ch, "スロットル", "Greens"),
    ],
    title=f"アクセルとブレーキ圧 - ラップ {int(selected_lap['num'])}",
)
show_fig(fig)